In [12]:
import pandas as pd
import numpy as np
from pathlib import Path

In [13]:
import igraph as ig

g = ig.Graph.Read_GraphML(
    "data/processed/broadway_network_clean.graphml"
)

print(g.summary())

IGRAPH U-W- 55261 1565292 -- 
+ attr: id (v), performer_name (v), weight (e)


In [14]:
coords = pd.read_csv(
    "data/layouts/coordinates/drl.csv"
)

coords.head()

,id,x,y
0,A_473388,166.447144,214.483383
1,A_34408,166.108566,213.783356
2,A_473389,-31.223625,227.400757
3,A_35813,166.484039,214.747192
4,A_438924,167.099854,214.465302


In [15]:
degree = pd.DataFrame({
    "id": g.vs["id"],
    "performer_name": g.vs["performer_name"],
    "degree": g.degree()
})

degree.head()

,id,performer_name,degree
0,A_473388,Dot Campbell,26
1,A_34408,Raymond Campbell,26
2,A_473389,Alice Carter,113
3,A_35813,Louis Cole,26
4,A_438924,Craddock and Shadney,26


In [16]:
nodes = coords.merge(
    degree,
    on="id",
    how="left"
)

nodes.head()

,id,x,y,performer_name,degree
0,A_473388,166.447144,214.483383,Dot Campbell,26
1,A_34408,166.108566,213.783356,Raymond Campbell,26
2,A_473389,-31.223625,227.400757,Alice Carter,113
3,A_35813,166.484039,214.747192,Louis Cole,26
4,A_438924,167.099854,214.465302,Craddock and Shadney,26


In [17]:
def linear_scale(x):
    return x

def log_scale(x):
    return np.log1p(x)

def sqrt_scale(x):
    return np.sqrt(x)

In [18]:
nodes["size_linear"] = linear_scale(nodes["degree"])

nodes["size_log"] = log_scale(nodes["degree"])

nodes["size_sqrt"] = sqrt_scale(nodes["degree"])

In [19]:
nodes[
    [
        "degree",
        "size_linear",
        "size_log",
        "size_sqrt"
    ]
].describe()

,degree,size_linear,size_log,size_sqrt
count,55261.000000,55261.000000,55261.000000,55261.000000
mean,56.650875,56.650875,3.629299,6.742954
std,62.159392,62.159392,0.931834,3.344197
min,0.000000,0.000000,0.000000,0.000000
25%,19.000000,19.000000,2.995732,4.358899
50%,37.000000,37.000000,3.637586,6.082763
75%,70.000000,70.000000,4.262680,8.366600
max,720.000000,720.000000,6.580639,26.832816


In [20]:
def rescale_sizes(series, min_size=2, max_size=20):
    return (
        (series - series.min())
        /
        (series.max() - series.min())
        *
        (max_size - min_size)
        + min_size
    )

In [21]:
for col in [
    "size_linear",
    "size_log",
    "size_sqrt"
]:
    nodes[col + "_px"] = rescale_sizes(
        nodes[col]
    )

In [22]:
nodes[
    [
        "size_linear_px",
        "size_log_px",
        "size_sqrt_px"
    ]
].describe()

,size_linear_px,size_log_px,size_sqrt_px
count,55261.000000,55261.000000,55261.000000
mean,3.416272,11.927210,6.523311
std,1.553985,2.548842,2.243356
min,2.000000,2.000000,2.000000
25%,2.475000,10.194216,4.924038
50%,2.925000,11.949877,6.080441
75%,3.750000,13.659694,7.612486
max,20.000000,20.000000,20.000000


In [23]:
output_dir = Path(
    "data/layouts/coordinates"
)

output_dir.mkdir(
    parents=True,
    exist_ok=True
)

In [24]:
nodes[
    [
        "id",
        "x",
        "y",
        "size_linear_px"
    ]
].rename(
    columns={
        "size_linear_px":"size"
    }
).to_csv(
    output_dir / "drl_linear.csv",
    index=False
)

In [25]:
nodes[
    [
        "id",
        "x",
        "y",
        "size_log_px"
    ]
].rename(
        columns={
            "size_log_px":"size"
        }
).to_csv(
    output_dir / "drl_log_size.csv",
    index=False
)

In [26]:
nodes[
    [
        "id",
        "x",
        "y",
        "size_sqrt_px"
    ]
].rename(
        columns={
            "size_sqrt_px":"size"
        }
).to_csv(
    output_dir / "drl_sqrt.csv",
    index=False
)

In [27]:
for file in [
    "drl_linear.csv",
    "drl_log_size.csv",
    "drl_sqrt.csv"
]:
    df = pd.read_csv(
        output_dir / file
    )
    
    print(
        file,
        df.shape,
        df["size"].min(),
        df["size"].max()
    )

drl_linear.csv (55261, 4) 2.0 20.0
drl_log_size.csv (55261, 4) 2.0 20.0
drl_sqrt.csv (55261, 4) 2.0 20.0


In [28]:
def rescale_sizes(series, min_size=2, max_size=15):
    return (
        (series - series.min())
        /
        (series.max() - series.min())
        *
        (max_size - min_size)
        + min_size
    )

In [29]:
nodes["size_linear_15"] = rescale_sizes(
    nodes["size_linear"]
)

nodes["size_log_15"] = rescale_sizes(
    nodes["size_log"]
)

nodes["size_sqrt_15"] = rescale_sizes(
    nodes["size_sqrt"]
)

In [30]:
nodes[
    [
        "size_linear_15",
        "size_log_15",
        "size_sqrt_15"
    ]
].describe()

,size_linear_15,size_log_15,size_sqrt_15
count,55261.000000,55261.000000,55261.000000
mean,3.022863,9.169652,5.266836
std,1.122322,1.840830,1.620201
min,2.000000,2.000000,2.000000
25%,2.343056,7.918045,4.111805
50%,2.668056,9.186022,4.946985
75%,3.263889,10.420890,6.053462
max,15.000000,15.000000,15.000000


In [31]:
output_dir = Path(
    "data/layouts/coordinates"
)

nodes[
    [
        "id",
        "performer_name",
        "x",
        "y",
        "size_sqrt_15"
    ]
].rename(
    columns={
        "size_sqrt_15": "size"
    }
).to_csv(
    output_dir / "drl_sqrt_15.csv",
    index=False
)

In [32]:
nodes[
    [
        "id",
        "performer_name",
        "x",
        "y",
        "size_log_15"
    ]
].rename(
    columns={
        "size_log_15": "size"
    }
).to_csv(
    output_dir / "drl_log_15.csv",
    index=False
)

In [33]:
nodes[
    [
        "id",
        "performer_name",
        "x",
        "y",
        "size_linear_15"
    ]
].rename(
    columns={
        "size_linear_15": "size"
    }
).to_csv(
    output_dir / "drl_linear_15.csv",
    index=False
)

# Overlap removal experiment

In [34]:
import pandas as pd
import numpy as np

nodes = pd.read_csv(
    "data/layouts/coordinates/drl_linear_15.csv"
)

nodes.head()

,id,performer_name,x,y,size
0,A_473388,Dot Campbell,166.447144,214.483383,2.469444
1,A_34408,Raymond Campbell,166.108566,213.783356,2.469444
2,A_473389,Alice Carter,-31.223625,227.400757,4.040278
3,A_35813,Louis Cole,166.484039,214.747192,2.469444
4,A_438924,Craddock and Shadney,167.099854,214.465302,2.469444


In [35]:
nodes["radius"] = nodes["size"] * 0.01

In [36]:
from scipy.spatial import cKDTree


def remove_overlaps(
    df,
    iterations=10,
    padding=0.02
):

    coords = df[["x", "y"]].values.copy()
    radii = df["radius"].values.copy()


    for i in range(iterations):

        tree = cKDTree(coords)

        pairs = tree.query_pairs(
            r=radii.max() * 2 + padding
        )


        displacement = np.zeros_like(coords)


        for a, b in pairs:

            delta = coords[a] - coords[b]

            distance = np.linalg.norm(delta)

            minimum = radii[a] + radii[b] + padding


            if distance < minimum:

                if distance == 0:
                    direction = np.random.randn(2)
                    direction /= np.linalg.norm(direction)

                else:
                    direction = delta / distance


                move = (minimum - distance) / 2

                displacement[a] += direction * move
                displacement[b] -= direction * move


        coords += displacement


    result = df.copy()

    result["x"] = coords[:,0]
    result["y"] = coords[:,1]

    return result

In [37]:
nodes_overlap = remove_overlaps(
    nodes,
    iterations=5
)

In [38]:
nodes_overlap[
    [
        "id",
        "performer_name",
        "x",
        "y",
        "size"
    ]
].to_csv(
    "data/layouts/coordinates/drl_linear_15_overlap.csv",
    index=False
)

In [42]:
def rescale_sizes(series, min_size=1, max_size=15):
    return (
        (series - series.min())
        /
        (series.max() - series.min())
        *
        (max_size - min_size)
        + min_size
    )

In [44]:
nodes["size_linear_1_15"] = rescale_sizes(
    nodes["size"],
    min_size=1,
    max_size=15
)

In [45]:
nodes[
    [
        "id",
        "performer_name",
        "x",
        "y",
        "size_linear_1_15"
    ]
].rename(
    columns={
        "size_linear_1_15": "size"
    }
).to_csv(
    "data/layouts/coordinates/drl_linear_1_15.csv",
    index=False
)